In [1]:
!pip install tensorflow

In [2]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, Input

In [3]:
from serial_manipulator_mixed_joints import SerialManipulator


In [4]:

# ============================================================
# Build 2R robot using the attached file
# ============================================================

def make_2r_robot_from_attached_file():
    l1 = 1.0
    l2 = 0.8
    m1 = 2.0
    m2 = 1.5
    lc1 = 0.5
    lc2 = 0.4
    I1 = 0.12
    I2 = 0.08
    g = 9.81

    DH_table = [
        [0.0, l1, 0.0, 0.0],
        [0.0, l2, 0.0, 0.0],
    ]
    joint_types = ["R", "R"]

    masses = [m1, m2]
    coms = [
        (lc1, 0.0, 0.0),
        (lc2, 0.0, 0.0),
    ]
    inertias = [
        np.diag([I1, I1, I1]),
        np.diag([I2, I2, I2]),
    ]

    # gravity along -y so the 2R planar arm in the xy-plane feels gravity
    robot = SerialManipulator.from_dh_table(
        DH_table,
        joint_types=joint_types,
        masses=masses,
        coms=coms,
        inertias=inertias,
        gravity=(0.0, -g, 0.0),
    )

    # attach parameters so your later true_coefficients_2R(...) code still works
    robot.l1 = l1
    robot.l2 = l2
    robot.m1 = m1
    robot.m2 = m2
    robot.lc1 = lc1
    robot.lc2 = lc2
    robot.I1 = I1
    robot.I2 = I2
    robot.g = g

    return robot

robot = make_2r_robot_from_attached_file()

# ============================================================
# Generate X and Y
# X columns: [theta1, theta2, theta1_dot, theta2_dot, theta1_ddot, theta2_ddot]
# Y columns: [tau1, tau2]
# ============================================================

np.random.seed(42)

n_samples = 10000

theta_values = np.random.uniform(0.0, 2.0 * np.pi, size=(n_samples, 2))
theta_dot_values = np.random.uniform(-5.0, 5.0, size=(n_samples, 2))
theta_doubledot_values = np.random.uniform(-5.0, 5.0, size=(n_samples, 2))

tau_values = np.stack([
    robot.inverse_dynamics(q, qdot, qddot)
    for q, qdot, qddot in zip(theta_values, theta_dot_values, theta_doubledot_values)
], axis=0)

X = np.concatenate(
    [theta_values, theta_dot_values, theta_doubledot_values],
    axis=1
).astype(np.float32)

Y = tau_values.astype(np.float32)

print("X shape:", X.shape)
print("Y shape:", Y.shape)
print("Sample X row:", X[0])
print("Sample Y row:", Y[0])

X shape: (10000, 6)
Y shape: (10000, 2)
Sample X row: [ 2.3533049  5.973514   2.299983  -3.15488   -2.0108795 -4.051822 ]
Sample Y row: [-81.395676 -27.971315]


In [5]:
def symmetric_upper_unweighted_rows_tf(x):
    x1 = x[:, 0:1]
    x2 = x[:, 1:2]
    return tf.concat([x1 * x1, x1 * x2, x2 * x2], axis=1)

def build_model(hidden_units=(64, 64, 64), activation="tanh"):
    # only q goes into the neural network
    inp_theta = Input(shape=(2,), name="theta_input")

    # extra = [qdot1, qdot2, qddot1, qddot2]
    inp_extra = Input(shape=(4,), name="extra_input")

    x = inp_theta
    for h in hidden_units:
        x = layers.Dense(h, activation=activation)(x)

    # 11 outputs:
    # [M11, M12, M22, g111, g121, g221, g112, g122, g222, G1, G2]
    params11 = layers.Dense(11, activation="linear", name="params11")(x)

    def physics_layer(inputs):
        p, e = inputs

        # split extra inputs
        qdot = e[:, 0:2]    # (batch,2)
        qddot = e[:, 2:4]   # (batch,2)

        # split learned parameters
        m_params = p[:, 0:3]
        c_params = p[:, 3:9]
        g_params = p[:, 9:11]

        # -------------------------
        # Mass matrix
        # -------------------------
        M11 = m_params[:, 0]
        M12 = m_params[:, 1]
        M22 = m_params[:, 2]

        row1 = tf.stack([M11, M12], axis=1)
        row2 = tf.stack([M12, M22], axis=1)
        M = tf.stack([row1, row2], axis=1)   # (batch,2,2)

        tau_m = tf.matmul(M, qddot[:, :, None])[:, :, 0]   # (batch,2)

        # -------------------------
        # Coriolis coefficient matrix
        # unweighted velocity basis
        # -------------------------
        vel_terms = symmetric_upper_unweighted_rows_tf(qdot)   # (batch,3)

        c11 = c_params[:, 0]
        c12 = c_params[:, 1]
        c13 = c_params[:, 2]
        c21 = c_params[:, 3]
        c22 = c_params[:, 4]
        c23 = c_params[:, 5]

        crow1 = tf.stack([c11, c12, c13], axis=1)
        crow2 = tf.stack([c21, c22, c23], axis=1)
        Cmat = tf.stack([crow1, crow2], axis=1)   # (batch,2,3)

        tau_c = tf.matmul(Cmat, vel_terms[:, :, None])[:, :, 0]   # (batch,2)

        # -------------------------
        # Gravity
        # -------------------------
        tau_g = g_params   # (batch,2)

        # total torque
        tau = tau_m + tau_c + tau_g
        return tau

    out = layers.Lambda(physics_layer, name="final_output")([params11, inp_extra])

    model = Model(inputs=[inp_theta, inp_extra], outputs=out)
    return model

In [6]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model, Input

# ============================================================
# Fixed train/val split for fair comparison
# ============================================================

np.random.seed(42)
tf.random.set_seed(42)

N = X.shape[0]
idx = np.random.permutation(N)
n_train = int(0.8 * N)

train_idx = idx[:n_train]
val_idx   = idx[n_train:]

X_train = X[train_idx].astype(np.float32)
Y_train = Y[train_idx].astype(np.float32)

X_val = X[val_idx].astype(np.float32)
Y_val = Y[val_idx].astype(np.float32)

# structured-model inputs
q_train_struct = X_train[:, :2]
extra_train    = X_train[:, 2:6]

q_val_struct = X_val[:, :2]
extra_val    = X_val[:, 2:6]

# DeLaN-style baseline inputs
q_train     = X_train[:, 0:2]
qdot_train  = X_train[:, 2:4]
qddot_train = X_train[:, 4:6]

q_val     = X_val[:, 0:2]
qdot_val  = X_val[:, 2:4]
qddot_val = X_val[:, 4:6]

# direct baseline input
x_train_direct = X_train[:, :6]
x_val_direct   = X_val[:, :6]


# ============================================================
# Keras-safe angle features
# ============================================================

def angle_features_tf(q):
    q1 = q[:, 0:1]
    q2 = q[:, 1:2]
    return tf.concat([
        q1, q2,
        tf.sin(q1), tf.cos(q1),
        tf.sin(q2), tf.cos(q2),
        tf.sin(q1 + q2), tf.cos(q1 + q2)
    ], axis=1)


# ============================================================
# Model 1: Your structured model
# ============================================================

model = build_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="mse",
    metrics=["mae"]
)

history1 = model.fit(
    [q_train_struct, extra_train],
    Y_train,
    validation_data=([q_val_struct, extra_val], Y_val),
    epochs=300,
    batch_size=64,
    verbose=1
)

param_model = Model(
    inputs=model.inputs,
    outputs=model.get_layer("params11").output
)


# ============================================================
# Model 2: DeLaN-inspired baseline
# ============================================================

class DeLaNInspired2R(tf.keras.Model):
    def __init__(self, hidden_units=64, eps=1e-3, fd_eps=1e-3, activation="tanh"):
        super().__init__()
        self.eps = eps
        self.fd_eps = fd_eps

        self.d1 = layers.Dense(hidden_units, activation=activation)
        self.d2 = layers.Dense(hidden_units, activation=activation)
        self.d3 = layers.Dense(hidden_units, activation=activation)

        # lower triangular parameters for 2x2 SPD M = L L^T + eps I
        # [l11_raw, l21, l22_raw]
        self.L_head = layers.Dense(3, activation="linear")

        # scalar potential energy
        self.V_head = layers.Dense(1, activation="linear")

    def shared_features(self, q, training=False):
        z = angle_features_tf(q)
        z = self.d1(z, training=training)
        z = self.d2(z, training=training)
        z = self.d3(z, training=training)
        return z

    def mass_matrix(self, q, training=False):
        z = self.shared_features(q, training=training)
        lp = self.L_head(z, training=training)

        l11_raw = lp[:, 0]
        l21     = lp[:, 1]
        l22_raw = lp[:, 2]

        l11 = tf.nn.softplus(l11_raw) + self.eps
        l22 = tf.nn.softplus(l22_raw) + self.eps

        zeros = tf.zeros_like(l11)

        row1 = tf.stack([l11, zeros], axis=1)
        row2 = tf.stack([l21, l22], axis=1)
        L = tf.stack([row1, row2], axis=1)  # (b,2,2)

        I = tf.eye(2, batch_shape=[tf.shape(q)[0]], dtype=q.dtype)
        M = tf.matmul(L, L, transpose_b=True) + self.eps * I
        return M

    def potential_energy(self, q, training=False):
        z = self.shared_features(q, training=training)
        V = self.V_head(z, training=training)   # (b,1)
        return V[:, 0]                          # (b,)

    def mass_and_potential(self, q, training=False):
        M = self.mass_matrix(q, training=training)
        V = self.potential_energy(q, training=training)
        return M, V

    def finite_difference_derivatives(self, q, training=False):
        """
        Returns:
            dM_dq: shape (b, 2, 2, 2) where dM_dq[:,:,:,k] = dM/dq_k
            dV_dq: shape (b, 2)
        """
        b = tf.shape(q)[0]
        e1 = tf.tile(tf.constant([[1.0, 0.0]], dtype=q.dtype), [b, 1])
        e2 = tf.tile(tf.constant([[0.0, 1.0]], dtype=q.dtype), [b, 1])

        h = tf.cast(self.fd_eps, q.dtype)

        q_p1 = q + h * e1
        q_m1 = q - h * e1
        q_p2 = q + h * e2
        q_m2 = q - h * e2

        M_p1, V_p1 = self.mass_and_potential(q_p1, training=training)
        M_m1, V_m1 = self.mass_and_potential(q_m1, training=training)

        M_p2, V_p2 = self.mass_and_potential(q_p2, training=training)
        M_m2, V_m2 = self.mass_and_potential(q_m2, training=training)

        dM_dq1 = (M_p1 - M_m1) / (2.0 * h)   # (b,2,2)
        dM_dq2 = (M_p2 - M_m2) / (2.0 * h)   # (b,2,2)

        dV_dq1 = (V_p1 - V_m1) / (2.0 * h)   # (b,)
        dV_dq2 = (V_p2 - V_m2) / (2.0 * h)   # (b,)

        dM_dq = tf.stack([dM_dq1, dM_dq2], axis=-1)     # (b,2,2,2)
        dV_dq = tf.stack([dV_dq1, dV_dq2], axis=1)      # (b,2)

        return dM_dq, dV_dq

    def coriolis_vector(self, q, qdot, training=False):
        """
        Christoffel form:
        c_ijk = 0.5 * (dM_ij/dq_k + dM_ik/dq_j - dM_jk/dq_i)
        tau_c_i = sum_j sum_k c_ijk qdot_j qdot_k
        """
        dM_dq, _ = self.finite_difference_derivatives(q, training=training)  # (b,i,j,k)

        term1 = dM_dq
        term2 = tf.transpose(dM_dq, [0, 1, 3, 2])
        term3 = tf.transpose(dM_dq, [0, 2, 3, 1])

        c = 0.5 * (term1 + term2 - term3)

        tau_c = tf.einsum("bijk,bj,bk->bi", c, qdot, qdot)  # (b,2)
        return tau_c

    def gravity_vector(self, q, training=False):
        _, dV_dq = self.finite_difference_derivatives(q, training=training)
        return dV_dq

    def call(self, inputs, training=False):
        q, qdot, qddot = inputs
        q = tf.cast(q, tf.float32)
        qdot = tf.cast(qdot, tf.float32)
        qddot = tf.cast(qddot, tf.float32)

        M = self.mass_matrix(q, training=training)
        tau_m = tf.matmul(M, qddot[:, :, None])[:, :, 0]
        tau_c = self.coriolis_vector(q, qdot, training=training)
        tau_g = self.gravity_vector(q, training=training)

        return tau_m + tau_c + tau_g


model2 = DeLaNInspired2R(hidden_units=64, eps=1e-3, fd_eps=1e-3, activation="tanh")

model2.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="mse",
    metrics=["mae"]
)

history2 = model2.fit(
    [q_train, qdot_train, qddot_train],
    Y_train,
    validation_data=([q_val, qdot_val, qddot_val], Y_val),
    epochs=300,
    batch_size=64,
    verbose=1
)


# ============================================================
# Model 3: Direct torque-learning baseline
# ============================================================

def build_direct_model(hidden_units=(64, 64, 64), activation="tanh"):
    inp = Input(shape=(6,), name="direct_input")

    q = inp[:, 0:2]
    qdot = inp[:, 2:4]
    qddot = inp[:, 4:6]

    q_feat = layers.Lambda(angle_features_tf, name="direct_angle_features")(q)
    x = layers.Concatenate()([q_feat, qdot, qddot])

    for h in hidden_units:
        x = layers.Dense(h, activation=activation)(x)

    out = layers.Dense(2, activation="linear", name="direct_tau_output")(x)

    return Model(inputs=inp, outputs=out)


model3 = build_direct_model(hidden_units=(64, 64, 64), activation="tanh")
model3.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="mse",
    metrics=["mae"]
)

history3 = model3.fit(
    x_train_direct,
    Y_train,
    validation_data=(x_val_direct, Y_val),
    epochs=300,
    batch_size=64,
    verbose=1
)


# ============================================================
# 4. Basic comparison of torque metrics
# ============================================================

Y_pred1 = model.predict([q_val_struct, extra_val], verbose=0)
Y_pred2 = model2.predict([q_val, qdot_val, qddot_val], verbose=0)
Y_pred3 = model3.predict(x_val_direct, verbose=0)

mae1 = np.mean(np.abs(Y_pred1 - Y_val))
mae2 = np.mean(np.abs(Y_pred2 - Y_val))
mae3 = np.mean(np.abs(Y_pred3 - Y_val))

rmse1 = np.sqrt(np.mean((Y_pred1 - Y_val) ** 2))
rmse2 = np.sqrt(np.mean((Y_pred2 - Y_val) ** 2))
rmse3 = np.sqrt(np.mean((Y_pred3 - Y_val) ** 2))

mae1_each = np.mean(np.abs(Y_pred1 - Y_val), axis=0)
mae2_each = np.mean(np.abs(Y_pred2 - Y_val), axis=0)
mae3_each = np.mean(np.abs(Y_pred3 - Y_val), axis=0)

print("\n" + "=" * 70)
print("INITIAL TORQUE METRICS")
print("=" * 70)
print("Structured model overall MAE :", mae1)
print("DeLaN-style model2 overall MAE:", mae2)
print("Direct model3 overall MAE    :", mae3)

print("Structured model overall RMSE :", rmse1)
print("DeLaN-style model2 overall RMSE:", rmse2)
print("Direct model3 overall RMSE    :", rmse3)

print("Structured model per-output MAE :", mae1_each)
print("DeLaN-style model2 per-output MAE:", mae2_each)
print("Direct model3 per-output MAE    :", mae3_each)


# ============================================================
# 5. Quick SPD checks (existing style)
# ============================================================

M_val_2 = model2.mass_matrix(tf.convert_to_tensor(q_val, dtype=tf.float32), training=False).numpy()

sym_err_2 = np.mean(np.abs(M_val_2 - np.transpose(M_val_2, (0, 2, 1))))
eigvals_2 = np.linalg.eigvalsh(M_val_2)
spd_rate_2 = np.mean(np.all(eigvals_2 > 0, axis=1))

print("\nmodel2 mean symmetry error:", sym_err_2)
print("model2 SPD rate:", spd_rate_2)

params11_val = param_model.predict([q_val_struct, extra_val], verbose=0)

m_params = params11_val[:, :3]
M11 = m_params[:, 0]
M12 = m_params[:, 1]
M22 = m_params[:, 2]

Nval = len(M11)
M_pred = np.zeros((Nval, 2, 2), dtype=np.float64)
M_pred[:, 0, 0] = M11
M_pred[:, 0, 1] = M12
M_pred[:, 1, 0] = M12
M_pred[:, 1, 1] = M22

sym_err_1 = np.mean(np.abs(M_pred - np.transpose(M_pred, (0, 2, 1))))
print("Model1 mean symmetry error:", sym_err_1)

eigvals_1 = np.linalg.eigvalsh(M_pred)
spd_mask_1 = np.all(eigvals_1 > 0, axis=1)
print("Model1 SPD rate (eig check):", np.mean(spd_mask_1))
print("Model1 minimum eigenvalue:", eigvals_1.min())

detM = M11 * M22 - M12**2
spd_mask_fast = (M11 > 0) & (detM > 0)
print("Model1 SPD rate (fast check):", np.mean(spd_mask_fast))
print("Model1 minimum M11:", M11.min())
print("Model1 minimum determinant:", detM.min())

# M_true = np.transpose(robot.mass_matrix(q_val_struct), (2, 0, 1))
M_true = np.stack([robot.mass_matrix(q) for q in q_val_struct], axis=0)
print("Model1 mass matrix MAE:", np.mean(np.abs(M_pred - M_true)))


# ============================================================
# 6. Comparison of physics-like parameters
# ============================================================

def true_coefficients_2R(robot, q):
    """
    q shape: (N, 2)

    returns shape: (N, 11)
    order:
    [M11, M12, M22, g111, g121, g221, g112, g122, g222, G1, G2]
    """
    q = np.asarray(q)
    theta1 = q[:, 0]
    theta2 = q[:, 1]

    l1 = robot.l1
    lc1 = robot.lc1
    lc2 = robot.lc2
    m1 = robot.m1
    m2 = robot.m2
    I1 = robot.I1
    I2 = robot.I2
    g = robot.g

    M11 = I1 + I2 + m1 * lc1**2 + m2 * (l1**2 + lc2**2 + 2 * l1 * lc2 * np.cos(theta2))
    M12 = I2 + m2 * (lc2**2 + l1 * lc2 * np.cos(theta2))
    M22 = (I2 + m2 * lc2**2) * np.ones_like(theta2)

    h = -m2 * l1 * lc2 * np.sin(theta2)

    g111 = np.zeros_like(theta2)
    g121 = 2 * h
    g221 = h

    g112 = -h
    g122 = np.zeros_like(theta2)
    g222 = np.zeros_like(theta2)

    G1 = (m1 * lc1 + m2 * l1) * g * np.cos(theta1) + m2 * lc2 * g * np.cos(theta1 + theta2)
    G2 = m2 * lc2 * g * np.cos(theta1 + theta2)

    coeffs = np.stack([
        M11, M12, M22,
        g111, g121, g221,
        g112, g122, g222,
        G1, G2
    ], axis=1)

    return coeffs


def true_mass_matrix_batch(robot, q):
    return np.transpose(robot.mass_matrix(q), (2, 0, 1))


def true_gravity_vector_batch(robot, q):
    return robot.gravity_vector(q).T


def true_coriolis_vector_batch(robot, q, qdot):
    return robot.coriolis_vector(q, qdot).T


# ============================================================
# 7. Utility metrics
# ============================================================

def compute_basic_metrics(pred, true):
    mae = np.mean(np.abs(pred - true))
    rmse = np.sqrt(np.mean((pred - true) ** 2))
    return mae, rmse


def compute_per_column_metrics(pred, true):
    mae_each = np.mean(np.abs(pred - true), axis=0)
    rmse_each = np.sqrt(np.mean((pred - true) ** 2, axis=0))
    mean_abs_true = np.mean(np.abs(true), axis=0)
    rel_mae_each = mae_each / (mean_abs_true + 1e-12)

    corr_each = []
    for k in range(true.shape[1]):
        t = true[:, k]
        p = pred[:, k]
        if np.std(t) < 1e-12 or np.std(p) < 1e-12:
            corr_each.append(np.nan)
        else:
            corr_each.append(np.corrcoef(t, p)[0, 1])
    corr_each = np.array(corr_each)

    return mae_each, rmse_each, rel_mae_each, corr_each


def print_metric_table(names, pred, true, title):
    mae_each, rmse_each, rel_mae_each, corr_each = compute_per_column_metrics(pred, true)

    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)
    print(f"{'Name':<10} {'MAE':>14} {'RMSE':>14} {'Rel_MAE':>14} {'Corr':>14}")
    print("-" * 90)
    for n, mae, rmse, rmae, corr in zip(names, mae_each, rmse_each, rel_mae_each, corr_each):
        print(f"{n:<10} {mae:14.6f} {rmse:14.6f} {rmae:14.6f} {corr:14.6f}")
    print("-" * 90)
    print(f"{'OVERALL':<10} {np.mean(np.abs(pred - true)):14.6f} {np.sqrt(np.mean((pred - true)**2)):14.6f}")
    print("=" * 90)


# ============================================================
# 8. Model1 evaluation: coefficients, M, SPD
# ============================================================

def evaluate_model1(model, param_model, robot, q_val_struct, extra_val):
    coeff_names = [
        "M11", "M12", "M22",
        "g111", "g121", "g221",
        "g112", "g122", "g222",
        "G1", "G2"
    ]

    print("\n" + "#" * 90)
    print("MODEL 1: STRUCTURED COEFFICIENT MODEL")
    print("#" * 90)

    Y_pred1 = model.predict([q_val_struct, extra_val], verbose=0)

    params11_val = param_model.predict([q_val_struct, extra_val], verbose=0)
    true11_val = true_coefficients_2R(robot, q_val_struct)

    print("params11_val shape:", params11_val.shape)
    print("true11_val   shape:", true11_val.shape)

    print_metric_table(
        coeff_names,
        params11_val,
        true11_val,
        title="Model1 coefficient comparison: predicted vs true"
    )

    m_params = params11_val[:, :3]
    M11 = m_params[:, 0]
    M12 = m_params[:, 1]
    M22 = m_params[:, 2]

    Nval = len(M11)
    M_pred = np.zeros((Nval, 2, 2), dtype=np.float64)
    M_pred[:, 0, 0] = M11
    M_pred[:, 0, 1] = M12
    M_pred[:, 1, 0] = M12
    M_pred[:, 1, 1] = M22

    M_true = true_mass_matrix_batch(robot, q_val_struct)

    sym_err = np.mean(np.abs(M_pred - np.transpose(M_pred, (0, 2, 1))))
    eigvals = np.linalg.eigvalsh(M_pred)
    spd_mask = np.all(eigvals > 0, axis=1)
    spd_rate = np.mean(spd_mask)

    detM = M11 * M22 - M12**2
    spd_mask_fast = (M11 > 0) & (detM > 0)

    print("\nModel1 mass matrix checks")
    print("-" * 50)
    print("Mean symmetry error           :", sym_err)
    print("SPD rate (eigenvalue check)   :", spd_rate)
    print("SPD rate (2x2 determinant)    :", np.mean(spd_mask_fast))
    print("Minimum eigenvalue            :", eigvals.min())
    print("Minimum M11                   :", M11.min())
    print("Minimum determinant           :", detM.min())
    print("Mass matrix MAE               :", np.mean(np.abs(M_pred - M_true)))
    print("Mass matrix RMSE              :", np.sqrt(np.mean((M_pred - M_true)**2)))

    qdot = extra_val[:, 0:2]
    vel_terms = np.stack([
        qdot[:, 0]**2,
        qdot[:, 0]*qdot[:, 1],
        qdot[:, 1]**2
    ], axis=1)

    C_pred = np.zeros((Nval, 2, 3), dtype=np.float64)
    C_pred[:, 0, 0] = params11_val[:, 3]
    C_pred[:, 0, 1] = params11_val[:, 4]
    C_pred[:, 0, 2] = params11_val[:, 5]
    C_pred[:, 1, 0] = params11_val[:, 6]
    C_pred[:, 1, 1] = params11_val[:, 7]
    C_pred[:, 1, 2] = params11_val[:, 8]

    coriolis_pred = (C_pred @ vel_terms[:, :, None]).squeeze(-1)
    coriolis_true = true_coriolis_vector_batch(robot, q_val_struct, qdot)

    gravity_pred = params11_val[:, 9:11]
    gravity_true = true_gravity_vector_batch(robot, q_val_struct)

    print_metric_table(
        ["tau_c1", "tau_c2"],
        coriolis_pred,
        coriolis_true,
        title="Model1 derived Coriolis vector comparison"
    )

    print_metric_table(
        ["G1", "G2"],
        gravity_pred,
        gravity_true,
        title="Model1 gravity vector comparison"
    )

    return {
        "Y_pred1": Y_pred1,
        "params11_val": params11_val,
        "true11_val": true11_val,
        "M_pred": M_pred,
        "M_true": M_true,
        "coriolis_pred": coriolis_pred,
        "coriolis_true": coriolis_true,
        "gravity_pred": gravity_pred,
        "gravity_true": gravity_true,
    }


# ============================================================
# 9. Model2 evaluation: M, G, C(q,qdot)qdot, SPD
# ============================================================

def evaluate_model2(model2, robot, q_val, qdot_val, qddot_val):
    print("\n" + "#" * 90)
    print("MODEL 2: DELAN-STYLE MODEL")
    print("#" * 90)

    Y_pred2 = model2.predict([q_val, qdot_val, qddot_val], verbose=0)

    M_pred2 = model2.mass_matrix(
        tf.convert_to_tensor(q_val, dtype=tf.float32),
        training=False
    ).numpy()

    M_true = true_mass_matrix_batch(robot, q_val)

    sym_err = np.mean(np.abs(M_pred2 - np.transpose(M_pred2, (0, 2, 1))))
    eigvals = np.linalg.eigvalsh(M_pred2)
    spd_mask = np.all(eigvals > 0, axis=1)
    spd_rate = np.mean(spd_mask)

    print("\nModel2 mass matrix checks")
    print("-" * 50)
    print("Mean symmetry error         :", sym_err)
    print("SPD rate                    :", spd_rate)
    print("Minimum eigenvalue          :", eigvals.min())
    print("Mass matrix MAE             :", np.mean(np.abs(M_pred2 - M_true)))
    print("Mass matrix RMSE            :", np.sqrt(np.mean((M_pred2 - M_true)**2)))

    gravity_pred2 = model2.gravity_vector(
        tf.convert_to_tensor(q_val, dtype=tf.float32),
        training=False
    ).numpy()

    gravity_true = true_gravity_vector_batch(robot, q_val)

    print_metric_table(
        ["G1", "G2"],
        gravity_pred2,
        gravity_true,
        title="Model2 gravity vector comparison"
    )

    coriolis_pred2 = model2.coriolis_vector(
        tf.convert_to_tensor(q_val, dtype=tf.float32),
        tf.convert_to_tensor(qdot_val, dtype=tf.float32),
        training=False
    ).numpy()

    coriolis_true = true_coriolis_vector_batch(robot, q_val, qdot_val)

    print_metric_table(
        ["tau_c1", "tau_c2"],
        coriolis_pred2,
        coriolis_true,
        title="Model2 Coriolis vector comparison"
    )

    coeff_like_pred2 = np.column_stack([
        M_pred2[:, 0, 0],
        M_pred2[:, 0, 1],
        M_pred2[:, 1, 1],
        gravity_pred2[:, 0],
        gravity_pred2[:, 1]
    ])

    true11_val = true_coefficients_2R(robot, q_val)
    coeff_like_true2 = np.column_stack([
        true11_val[:, 0],
        true11_val[:, 1],
        true11_val[:, 2],
        true11_val[:, 9],
        true11_val[:, 10],
    ])

    print_metric_table(
        ["M11", "M12", "M22", "G1", "G2"],
        coeff_like_pred2,
        coeff_like_true2,
        title="Model2 coefficient-like comparison (M and G only)"
    )

    return {
        "Y_pred2": Y_pred2,
        "M_pred2": M_pred2,
        "M_true": M_true,
        "gravity_pred2": gravity_pred2,
        "gravity_true": gravity_true,
        "coriolis_pred2": coriolis_pred2,
        "coriolis_true": coriolis_true,
    }


# ============================================================
# 10. Model3 evaluation: torque only
# ============================================================

def evaluate_model3(model3, x_val_direct):
    print("\n" + "#" * 90)
    print("MODEL 3: DIRECT TORQUE LEARNING MODEL")
    print("#" * 90)

    Y_pred3 = model3.predict(x_val_direct, verbose=0)

    return {
        "Y_pred3": Y_pred3
    }


# ============================================================
# 11. Compare torque prediction of all three models
# ============================================================

def compare_model_torques(model1_pred, model2_pred, model3_pred, Y_val):
    print("\n" + "#" * 90)
    print("FINAL TORQUE COMPARISON: MODEL1 vs MODEL2 vs MODEL3")
    print("#" * 90)

    mae1, rmse1 = compute_basic_metrics(model1_pred, Y_val)
    mae2, rmse2 = compute_basic_metrics(model2_pred, Y_val)
    mae3, rmse3 = compute_basic_metrics(model3_pred, Y_val)

    mae1_each = np.mean(np.abs(model1_pred - Y_val), axis=0)
    mae2_each = np.mean(np.abs(model2_pred - Y_val), axis=0)
    mae3_each = np.mean(np.abs(model3_pred - Y_val), axis=0)

    print("Structured model overall MAE   :", mae1)
    print("DeLaN-style model overall MAE  :", mae2)
    print("Direct model overall MAE       :", mae3)

    print("Structured model overall RMSE  :", rmse1)
    print("DeLaN-style model overall RMSE :", rmse2)
    print("Direct model overall RMSE      :", rmse3)

    print("Structured model per-output MAE  :", mae1_each)
    print("DeLaN-style model per-output MAE :", mae2_each)
    print("Direct model per-output MAE      :", mae3_each)





Epoch 1/300
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 983.1458 - mae: 23.2708 - val_loss: 701.2419 - val_mae: 20.3814
Epoch 2/300
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 624.3634 - mae: 18.9959 - val_loss: 492.8269 - val_mae: 17.2868
Epoch 3/300
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 466.8988 - mae: 16.5372 - val_loss: 411.1051 - val_mae: 15.7605
Epoch 4/300
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 391.1773 - mae: 15.2251 - val_loss: 356.9915 - val_mae: 14.6865
Epoch 5/300
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 341.1725 - mae: 14.2143 - val_loss: 309.7005 - val_mae: 13.6254
Epoch 6/300
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 300.5480 - mae: 13.2308 - val_loss: 268.2065 - val_mae: 12.5234
Epoch 7/300
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 263.2727 - mae: 12.2064 - val_loss: 232.9504 - val_mae: 11.4468
Epoch 8/300
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 230.5723 - mae: 11.2480 - val_loss: 202.9494 - val_mae: 10.5142

In [10]:
def true_mass_matrix_batch(robot, q):
    import numpy as np
    
    M_list = []
    
    for qi in q:
        M = robot.mass_matrix(qi)
        M_list.append(M)
        
    return np.array(M_list)

In [11]:
def compute_per_column_metrics(pred, true):
    mae_each = np.mean(np.abs(pred - true), axis=0)
    rmse_each = np.sqrt(np.mean((pred - true) ** 2, axis=0))
    mean_abs_true = np.mean(np.abs(true), axis=0)
    rel_mae_each = mae_each / (mean_abs_true + 1e-12)

    corr_each = []
    for k in range(true.shape[1]):
        t = true[:, k]
        p = pred[:, k]
        if np.std(t) < 1e-12 or np.std(p) < 1e-12:
            corr_each.append(np.nan)
        else:
            corr_each.append(np.corrcoef(t, p)[0, 1])
    corr_each = np.array(corr_each)

    return mae_each, rmse_each, rel_mae_each, corr_each

In [15]:
def true_coriolis_vector_batch(robot, q, qdot):
    import numpy as np
    
    n = q.shape[0]   # number of samples
    dof = robot.n    # number of joints
    
    return np.zeros((n, dof))

In [17]:
def true_gravity_vector_batch(robot, q):
    import numpy as np
    
    G_list = []
    
    for qi in q:
        G = robot.gravity_vector(qi)
        G_list.append(G)
        
    return np.array(G_list)

In [18]:
# ============================================================
# 12. Run everything
# ============================================================

results1 = evaluate_model1(
    model=model,
    param_model=param_model,
    robot=robot,
    q_val_struct=q_val_struct,
    extra_val=extra_val
)

results2 = evaluate_model2(
    model2=model2,
    robot=robot,
    q_val=q_val,
    qdot_val=qdot_val,
    qddot_val=qddot_val
)

results3 = evaluate_model3(
    model3=model3,
    x_val_direct=x_val_direct
)

compare_model_torques(
    model1_pred=results1["Y_pred1"],
    model2_pred=results2["Y_pred2"],
    model3_pred=results3["Y_pred3"],
    Y_val=Y_val
)


##########################################################################################
MODEL 1: STRUCTURED COEFFICIENT MODEL
##########################################################################################
params11_val shape: (2000, 11)
true11_val   shape: (2000, 11)

Model1 coefficient comparison: predicted vs true
Name                  MAE           RMSE        Rel_MAE           Corr
------------------------------------------------------------------------------------------
M11              5.894004       6.137127       2.434621       0.999869
M12              1.904381       2.088905       4.365053       0.999805
M22              1.924480       1.924555       6.013890            nan
g111             0.010709       0.013424 10709064704.000000            nan
g121             1.499404       1.670473       1.995105       0.999945
g221             0.752536       0.837749       2.002647       0.999883
g112             0.751492       0.836571       1.999868       0.999910
g122